# Day 0: Python File Operations

A hands-on introduction to reading and writing text (`.txt`), CSV (`.csv`), and JSON (`.json`) files. These are the formats used by the Day 3 streaming exercise.

**How to work:** read each note, predict what the code will do, run the cell, then answer the prompt. Examples use a dedicated `day0_file_practice` folder so they do not overwrite your project data.

**Time:** about 60–90 minutes. Choose the `.venv` kernel in VS Code and run cells from top to bottom.

## 1. Paths and safe file handling

A path names a file or folder. `pathlib.Path` builds paths portably. Open files with `with`: Python closes the file automatically when the block ends, including when an error occurs.

Modes you will commonly see: `"r"` read, `"w"` write (replace existing contents), `"a"` append, and `"x"` create only if the file does not already exist. Text is the default mode.

In [1]:
from pathlib import Path

workspace = Path("day0_file_practice")
workspace.mkdir(exist_ok=True)
notes_path = workspace / "notes.txt"
print(notes_path)
print("Exists before creating?", notes_path.exists())

day0_file_practice\notes.txt
Exists before creating? False


**In your words:** What does the `with` block do for an opened file?

> Write here.

**Why might mode `"w"` be dangerous if the file already has information?**

> Write here.

---

## 2. Text files (`.txt`)

Use `write_text` / `read_text` for small text files. For larger files, use `open` and process incrementally. Specify `encoding="utf-8"` for predictable text handling. `write_text` replaces the file; `open(..., "a")` appends. The `newline` option is usually not needed for plain text.

In [2]:
notes_path.write_text("First line\nSecond line\n", encoding="utf-8")
print(notes_path.read_text(encoding="utf-8"))

with notes_path.open("a", encoding="utf-8") as file:
    file.write("Appended line\n")

with notes_path.open("r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        print(line_number, repr(line))

First line
Second line

1 'First line\n'
2 'Second line\n'
3 'Appended line\n'


Notice that lines read from a text file usually include the newline character `\n`; `strip()` can remove whitespace, while `rstrip("\n")` removes only the line ending. Avoid `read_text()` or `list(file)` for very large files because they load all content into memory.

**Practice:** Write `hello.txt` with your own two lines, read it back, then append a third line.

```python
# Your code here
```

---

## 3. CSV files with the `csv` module

CSV is tabular text. Use `csv.DictReader` to read rows as dictionaries keyed by the header. Use `csv.DictWriter` to write dictionaries. Open CSV files with `newline=""` so the csv module handles line endings consistently. CSV values are strings when read; convert values to `int`, `float`, dates, etc. yourself.

In [ ]:
import csv

orders_path = workspace / "orders.csv"
orders = [
    {"order_id": 101, "customer_id": "C001", "amount": 12.5},
    {"order_id": 102, "customer_id": "C002", "amount": 8.0},
]

In [ ]:
with orders_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["order_id", "customer_id", "amount"])
    writer.writeheader()
    writer.writerows(orders)

with orders_path.open("r", newline="", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        print(row)

Observe the worked example: `DictWriter` writes dictionaries as CSV rows; `DictReader` turns each CSV row into a dictionary.

**What type is `row["order_id"]` after CSV reading? Why?**

> Write here.

Try changing a value and rerunning both cells.

---

### CSV: read and convert one row at a time

`csv.DictReader` is an iterator: each loop gets the next row, rather than requiring all rows to be stored in a list. Convert fields at the point you need their typed values. Use `try` / `except ValueError` for bad numeric input.

In [ ]:
# Run after completing the CSV write/read exercise above.
with orders_path.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        order_id = int(row["order_id"])
        amount = float(row["amount"])
        print(order_id, row["customer_id"], amount)

**Practice:** Add a row whose amount is `"not-a-number"`. Change the reader to catch `ValueError` and print a useful message while continuing to later rows.

```python
# Your code here
```

---

## 4. JSON files with the `json` module

JSON maps naturally to Python dictionaries, lists, strings, numbers, booleans, and `None`. Use `json.dump(value, file)` to write to an open file; use `json.load(file)` to read. `json.dumps` and `json.loads` work with JSON strings rather than files. `indent=2` makes output readable. JSON object keys are strings.

In [ ]:
import json

customer = {
    "customer_id": "C001",
    "active": True,
    "tags": ["new", "email"],
    "preferences": None,
}
customer_path = workspace / "customer.json"

with customer_path.open("w", encoding="utf-8") as file:
    json.dump(customer, file, indent=2)

print(customer_path.read_text(encoding="utf-8"))

In [ ]:
with customer_path.open("r", encoding="utf-8") as file:
    loaded_customer = json.load(file)

print(loaded_customer)
print(type(loaded_customer), type(loaded_customer["tags"]))

json_text = json.dumps({"ok": True, "count": 3})
print(json_text)
print(json.loads(json_text))

**Practice:** Add a list of two orders to a Python dictionary, save it as `orders.json`, load it again, and print the amount of the second order.

```python
# Your code here
```

---

## 5. JSON Lines (`.jsonl`)

JSON Lines stores one complete JSON value per line, often one object per line. This format works well for streaming because you can read and write a record at a time. A JSONL file is not one large JSON array: do not call `json.load` once on the whole file. Instead, parse each line separately with `json.loads`.

In [ ]:
jsonl_path = workspace / "orders.jsonl"

with jsonl_path.open("w", encoding="utf-8") as file:
    for order in orders:
        file.write(json.dumps(order) + "\n")

with jsonl_path.open("r", encoding="utf-8") as file:
    for line in file:
        record = json.loads(line)
        print(record)

**Explain the difference:** What shape does a `.json` file usually contain here, and how is a `.jsonl` file organized?

> Write here.

**Practice:** Count the JSONL records without loading them all into a list.

```python
# Your code here
```

---

## 6. Errors and robust file handling

Common file-related errors include `FileNotFoundError`, `PermissionError`, `UnicodeDecodeError`, `csv.Error`, and `json.JSONDecodeError`. Catch errors you can handle meaningfully. Avoid catching every exception without a plan. Use `try` / `except` / `else` / `finally` for clear control flow; `with` already handles closing an opened file.

In [ ]:
bad_json_path = workspace / "bad.json"
bad_json_path.write_text("{not valid json", encoding="utf-8")

try:
    with bad_json_path.open("r", encoding="utf-8") as file:
        value = json.load(file)
except json.JSONDecodeError as error:
    print(f"Could not parse JSON: {error}")
else:
    print("Loaded:", value)
finally:
    print("Finished reading attempt")

**In your words:** What is the difference between a missing file and malformed JSON? Which exception would you expect for each?

> Write here.

---

## 7. Large files: stream instead of collecting

For large inputs, open the file and process each row or line inside a loop. `rows = list(csv.DictReader(file))` stores every parsed row in memory. Streaming keeps memory usage much lower. Keep both input and output files open with `with` blocks while processing.

In [ ]:
# Small demonstration of streaming CSV rows.
row_count = 0
amount_total = 0.0
with orders_path.open("r", newline="", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        row_count += 1
        amount_total += float(row["amount"])
print("Rows:", row_count, "Amount total:", amount_total)

**Before Day 3 P1, explain these from memory:**

1. Why use `with open(...)`?
2. Why pass `newline=""` when opening CSV?
3. Why are CSV values converted explicitly?
4. How do you write one JSON object per output line?
5. Why avoid building `list(reader)` for a million-row file?

> Write your answers here.

## Completion check

- [ ] I can read and write text files with UTF-8 and `with`.
- [ ] I can write/read CSV with `DictWriter` and `DictReader`.
- [ ] I know CSV fields are strings until converted.
- [ ] I can read and write JSON with `json.load` and `json.dump`.
- [ ] I can process JSON Lines one line at a time.
- [ ] I know how to handle a malformed JSON record and a missing file.
- [ ] I can explain why streaming is better for large files.

**Confidence before P1 (1–5):** ___ / 5

**Questions for review:**

> Write here.